# Oil Price Forecasting

## Project Overview

Forecasts the **daily crude oil price** (USD per barrel) over a 14-day horizon. Synthetic data spans 2 years with supply/demand cycles, geopolitical shocks, and seasonal demand patterns.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily oil prices, predict the next 14 days. Oil price forecasts are critical for energy companies, airlines, governments, and financial markets — oil is the world's most-traded commodity.

## Why This Project Matters

Oil prices affect virtually every sector of the economy. Transportation, manufacturing, agriculture, and consumer goods all depend on energy costs. Oil price shocks trigger recessions, inflation, and geopolitical conflicts. Even modest forecast improvements have enormous economic value.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily crude oil prices
- Mean-reverting around $75 with supply/demand cycles
- Geopolitical supply shocks (spikes)
- Seasonal demand pattern (winter heating, summer driving)
- High volatility with clustering

| Column | Description |
|--------|-------------|
| `date` | Date |
| `oil_usd_bbl` | Daily crude oil price (USD per barrel) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'oil_usd_bbl'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

# Mean-reverting around $75 with slow cycle
equilibrium = 75 + 10 * np.sin(2 * np.pi * t / 500)
price = np.zeros(N_POINTS)
price[0] = 75

theta = 0.03  # mean reversion
for i in range(1, N_POINTS):
    vol = 1.5 + 0.5 * np.abs(np.sin(2 * np.pi * i / 200))  # volatility clustering
    price[i] = price[i-1] + theta * (equilibrium[i] - price[i-1]) + vol * np.random.normal()

# Seasonal demand (winter heating + summer driving)
seasonal = 3 * np.sin(2 * np.pi * (t - 15) / 365) + 2 * np.sin(2 * np.pi * (t - 180) / 365)
price += seasonal

# Geopolitical supply shocks
for s in range(0, N_POINTS, 200):
    shock_day = min(s + np.random.randint(50, 150), N_POINTS - 8)
    spike = np.random.uniform(8, 20) * np.random.choice([-1, 1])
    for j in range(min(8, N_POINTS - shock_day)):
        price[shock_day + j] += spike * np.exp(-0.25 * j)

oil_usd_bbl = np.maximum(price, 30).round(2)

df = pd.DataFrame({'date': dates, 'oil_usd_bbl': oil_usd_bbl})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,oil_usd_bbl
0,2022-01-01,74.15
1,2022-01-02,74.92
2,2022-01-03,74.71
3,2022-01-04,75.72
4,2022-01-05,78.09


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('oil_usd_bbl Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("oil_usd_bbl Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

oil_usd_bbl Statistics:
count    730.000000
mean      76.066397
std        8.725953
min       56.200000
25%       68.990000
50%       76.205000
75%       81.977500
max      102.900000
Name: oil_usd_bbl, dtype: float64

CV: 0.115
Skewness: 0.222


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:        2.6 | RMSE:        3.3 | MAPE:  3.43%
Seasonal Naive (7)             | MAE:        2.9 | RMSE:        3.2 | MAPE:  3.82%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                          Adjusted R-Squared    R-Squared       RMSE  Time Taken
Model                                                                           
KernelRidge                     23071.460281 -1773.650791  75.643521    0.050499
GaussianProcessRegressor         5250.429956  -402.802304  36.082754    0.035155
DecisionTreeRegressor             824.739748   -62.364596  14.293501    0.011551
MLPRegressor                      509.377369   -38.105951  11.228887    0.231319
XGBRegressor                      280.110645   -20.470050   8.320162    0.427488
BaggingRegressor                  225.520936   -16.270841   7.462288    0.047956
RandomForestRegressor             111.597626    -7.507510   5.237409    0.410579
CatBoostRegressor                  56.223811    -3.247985   3.700895    1.306640
ExtraTreesRegressor                54.316019    -3.101232   3.636407    0.220466
ExtraTreeRegressor                 50.593609    -2.814893   3.507167    0.008068


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: lgbm
FLAML (lgbm)                   | MAE:        1.8 | RMSE:        2.0 | MAPE:  2.32%
FLAML Test (lgbm)              | MAE:        1.2 | RMSE:        1.6 | MAPE:  1.61%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:        2.0 | RMSE:        2.1 | MAPE:  2.60%
SF AutoETS                     | MAE:        1.9 | RMSE:        2.0 | MAPE:  2.49%
SF SeasonalNaive               | MAE:        1.9 | RMSE:        2.2 | MAPE:  2.47%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
             model      MAE     RMSE     MAPE
 FLAML Test (lgbm) 1.241357 1.611811 1.613516
      FLAML (lgbm) 1.780404 2.047484 2.318326
        SF AutoETS 1.914903 1.996852 2.491588
  SF SeasonalNaive 1.915000 2.168148 2.469953
      SF AutoARIMA 1.995394 2.078391 2.601644
Naive (Last Value) 2.586429 3.256996 3.427086
Seasonal Naive (7) 2.924286 3.220561 3.816808

Top 3:
            model      MAE     RMSE     MAPE
FLAML Test (lgbm) 1.241357 1.611811 1.613516
     FLAML (lgbm) 1.780404 2.047484 2.318326
       SF AutoETS 1.914903 1.996852 2.491588


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 0.20, Std: 1.60


## Interpretation and Business Insight

- Oil price mean-reverts around a slowly shifting equilibrium
- Supply/demand shocks create temporary price dislocations
- Seasonal demand patterns (winter heating, summer driving) are mild but persistent
- Volatility clusters — turbulent periods follow calm ones
- Mean reversion is stronger than in gold, making forecasting slightly easier

## Limitations

1. Synthetic — real oil depends on OPEC decisions, shale production, global GDP
2. No supply-side data (rig counts, OPEC quotas, strategic reserves)
3. No demand-side data (refinery runs, GDP, travel activity)
4. No cross-commodity relationships (natural gas, renewables)
5. No geopolitical event data

## How to Improve This Project

1. Add supply/demand fundamentals (inventories, rig counts, refinery data)
2. Include macro indicators (USD, global PMI, shipping indices)
3. Use GARCH for volatility modeling
4. Add OPEC meeting dates and policy announcements
5. Test regime-switching models for structural breaks

## Production Considerations

- Daily price forecasting for energy procurement
- Airline fuel hedging optimization
- Government budget and fiscal planning
- Integration with real-time market data APIs

## Common Mistakes

1. Ignoring mean-reversion in oil (it's stronger than in most assets)
2. Not modeling supply shocks as jump processes
3. Using symmetric models for asymmetric price behavior
4. Forecasting without accounting for OPEC policy regime
5. Confusing correlation with causation in macro relationships

## Mini Challenge / Exercises

1. Test for mean-reversion using the ADF test
2. Decompose into trend + seasonal + residual
3. Build a volatility forecast using rolling standard deviation
4. Compare forecast accuracy in calm vs turbulent regimes

## Final Summary / Key Takeaways

1. Oil price forecasting is critical for the global economy
2. Mean-reversion is a useful property for forecasting models
3. Supply shocks are the primary source of forecast error
4. Seasonal demand patterns provide a predictable component
5. Real deployment needs fundamental supply/demand data

In [18]:
import json
metrics = {
    'project': 'Oil Price Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Oil Price Forecasting — Complete ===")

Metrics saved.

=== Oil Price Forecasting — Complete ===
